# AIMO3 v14 — Robust TIR + Domain Routing

Modeled directly after the 43/50 top notebook pattern.
No meta-harness search overhead — uses proven domain-routing retrieval.

In [ ]:
# Cell 0: Uninstall conflicting packages + install vLLM
# Exact pattern from andreasbis/aimo-3-gpt-oss-120b-with-tools
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow' 2>/dev/null

In [ ]:
import os
import sys
import subprocess

def set_env(input_archive, temp_dir):
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install',
        '--no-index', '--find-links', f'{temp_dir}/wheels',
        'vllm', 'openai_harmony'
    ], check=True)

set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz',
    temp_dir='/kaggle/tmp/setup'
)

# Set tiktoken path for offline usage
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'

print('Setup complete.')

In [ ]:
# Cell 2: Imports
import os, sys, time, re, json, traceback, threading, gc, math, glob, random
from io import StringIO
from contextlib import redirect_stdout, redirect_stderr
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import polars as pl
import httpx
import kaggle_evaluation.aimo_3_inference_server

# Config
MODEL_PATH = '/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1'
VLLM_PORT = 8000
SERVED_NAME = 'gpt-oss'
VLLM_URL = f'http://0.0.0.0:{VLLM_PORT}/v1'
ANSWER_MOD = 100_000
N_SAMPLES = 16
MAX_TOKENS = 8192
TEMPERATURE = 0.7
CODE_TIMEOUT = 30
TIR_MAX_RETRIES = 2
NOTEBOOK_LIMIT = 17400  # ~4.8 hours safe limit
BASE_TIMEOUT = 270      # min seconds per problem
HIGH_TIMEOUT = 600      # max seconds per problem

print(f'Model: {MODEL_PATH}')
print(f'N_SAMPLES={N_SAMPLES}, MAX_TOKENS={MAX_TOKENS}')

In [ ]:
# Cell 3: Pre-load model weights + start vLLM server
print(f'Pre-loading weights from {MODEL_PATH}...')
t0 = time.time()
files_to_load = []
for root, _, files in os.walk(MODEL_PATH):
    for fn in files:
        fp = os.path.join(root, fn)
        if os.path.isfile(fp): files_to_load.append(fp)

def _read(p):
    with open(p, 'rb') as f:
        while f.read(1024*1024*1024): pass
with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(_read, files_to_load))
print(f'Pre-loaded {len(files_to_load)} files in {time.time()-t0:.1f}s')

cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_PATH,
    '--served-model-name', SERVED_NAME,
    '--tensor-parallel-size', '1',
    '--gpu-memory-utilization', '0.99',
    '--dtype', 'auto',
    '--kv-cache-dtype', 'fp8_e4m3',
    '--max-model-len', '81920',
    '--max-num-batched-tokens', '2048',
    '--max-num-seqs', '64',
    '--host', '0.0.0.0',
    '--port', str(VLLM_PORT),
    '--enable-prefix-caching',
    '--disable-log-stats',
    '--trust-remote-code',
]

log_file = open('vllm_server.log', 'w')
server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True)

print('Waiting for vLLM server...')
for i in range(300):
    try:
        r = httpx.get(f'{VLLM_URL}/models', timeout=2)
        if r.status_code == 200:
            print(f'Server ready in {i}s')
            break
    except: pass
    rc = server_proc.poll()
    if rc is not None:
        log_file.flush()
        with open('vllm_server.log') as f: print(f.read()[-3000:])
        raise RuntimeError(f'vLLM died with code {rc}')
    time.sleep(1)
else:
    log_file.flush()
    with open('vllm_server.log') as f: print(f.read()[-3000:])
    raise RuntimeError('vLLM timeout')

print('vLLM server ready.')

In [ ]:
# Cell 4: vLLM client + answer extraction + code sandbox

def vllm_chat(prompt, n=1, temp=0.7, max_tok=4096, stop=None):
    """Call vLLM server."""
    payload = {'model': SERVED_NAME, 'temperature': temp, 'max_tokens': max_tok, 'n': n,
               'messages': [{'role': 'user', 'content': prompt}]}
    if stop: payload['stop'] = stop
    r = httpx.post(f'{VLLM_URL}/chat/completions', json=payload, timeout=300)
    r.raise_for_status()
    return [c['message']['content'] or '' for c in r.json()['choices']]

# Answer extraction
def _pn(s):
    s = s.strip().replace('\\,','').replace('\\;','').replace(',','')
    s = re.sub(r'\\text\{.*?\}','',s); s = re.sub(r'\\mathrm\{.*?\}','',s)
    try: return int(s)
    except: pass
    try:
        f=float(s)
        if f==int(f) and f==f: return int(f)
    except: pass
    m = re.search(r'(-?\d+(?:\.\d+)?)', s)
    if m:
        try:
            f=float(m.group(1))
            if f==int(f): return int(f)
        except: pass
    return None

def extract_boxed(text):
    idx = text.rfind('\\boxed')
    if idx == -1: return None
    bs = text.find('{', idx)
    if bs == -1: return None
    d, e = 0, bs
    for i in range(bs, len(text)):
        if text[i]=='{': d+=1
        elif text[i]=='}':
            d-=1
            if d==0: e=i; break
    return _pn(text[bs+1:e].strip())

def extract_answer(text):
    r = extract_boxed(text)
    if r is not None: return r % ANSWER_MOD
    for ms in [re.findall(r'```output\s*(.*?)```',text,re.DOTALL)]:
        if ms:
            r=_pn(ms[-1].strip())
            if r is not None: return r % ANSWER_MOD
    for p in [r'answer\s+is\s*[:\s]*(-?\d+)',r'answer\s*[=:]\s*(-?\d+)']:
        ms=re.findall(p,text,re.IGNORECASE)
        if ms:
            r=_pn(ms[-1])
            if r is not None: return r % ANSWER_MOD
    ms=re.findall(r'(?<![.\d])(-?\d+)(?![.\d])',text)
    if ms:
        r=_pn(ms[-1])
        if r is not None: return r % ANSWER_MOD
    return None

# Code sandbox
SIMPORTS = 'import math,numpy as np,sympy as sp\nfrom sympy import *\nfrom itertools import combinations,permutations,product as iproduct\nfrom collections import Counter,defaultdict\nfrom fractions import Fraction\nimport itertools\n'
def exec_code(code, timeout=CODE_TIMEOUT):
    so,se=StringIO(),StringIO()
    ns={}
    try: exec(SIMPORTS,ns)
    except: pass
    ex=[None]
    def _r():
        try:
            with redirect_stdout(so),redirect_stderr(se): exec(code,ns)
        except Exception as e: ex[0]=e
    t=threading.Thread(target=_r,daemon=True); t.start(); t.join(timeout=timeout)
    if t.is_alive(): return '','Timeout',False
    if ex[0]: return so.getvalue()or'',''.join(traceback.format_exception(type(ex[0]),ex[0],ex[0].__traceback__)),False
    return so.getvalue()or'',se.getvalue()or'',True

def get_code_blocks(text):
    b=re.findall(r'```python\s*\n(.*?)```',text,re.DOTALL)
    return b if b else re.findall(r'```\s*\n(.*?)```',text,re.DOTALL)

print('Utilities ready.')

In [ ]:
# Cell 5: Domain-routing prompt builder (proven Meta-Harness pattern)

GEO_KW = ['triangle','circle','angle','perpendicular','inscribed','tangent','polygon','circumscri','midpoint','altitude']
NT_KW = ['prime','divisible','modulo','gcd','remainder','congruent','coprime','fermat','euler','residue']
COMBO_KW = ['how many','number of ways','permutation','combinat','probability','expected value','pigeonhole','coloring']

def classify(q):
    q=q.lower()
    s={'geometry':sum(1 for k in GEO_KW if k in q),
       'number_theory':sum(1 for k in NT_KW if k in q),
       'combinatorics':sum(1 for k in COMBO_KW if k in q)}
    for d in ['combinatorics','geometry','number_theory']:
        if s[d]>=2: return d
    best=max(s,key=s.get)
    return best if s[best]>=1 else 'algebra'

SYSTEM = '''You are a world-class IMO competitor. Solve step by step.
- Write Python code in ```python ... ``` blocks.
- After code runs you see output in ```output ... ```.
- Use sympy, numpy. Verify with code.
- Final answer in \\boxed{N}, integer 0-99999.'''

HINTS = {
    'algebra': 'Use sympy.symbols() and sympy.solve(). Verify by substitution.',
    'combinatorics': 'Enumerate small cases first. Use itertools. Check with brute force.',
    'geometry': 'Set up coordinates. Use sympy geometry. Verify numerically.',
    'number_theory': 'Use sympy.factorint(), pow(b,e,m). Check small cases first.',
}

def build_prompt(problem):
    domain = classify(problem)
    hint = HINTS.get(domain, 'Solve step by step with code.')
    return f"{SYSTEM}\n\n{hint}\n\nProblem:\n{problem}\n\nSolution:", domain

print('Prompt builder ready.')

In [ ]:
# Cell 6: TIR solver + voting

def solve_problem(problem_text):
    """Solve one problem: generate N solutions with TIR, vote on answer."""
    prompt, domain = build_prompt(problem_text)
    
    # Generate N completions
    try:
        responses = vllm_chat(prompt, n=N_SAMPLES, temp=TEMPERATURE,
                              max_tok=MAX_TOKENS, stop=['```output'])
    except Exception as e:
        print(f'  vLLM error: {e}')
        return 0
    
    results = []
    for text in responses:
        full = text
        code_ok = False
        
        # TIR: execute code blocks
        blocks = get_code_blocks(text)
        if blocks:
            so, se, ok = exec_code(blocks[-1])
            code_ok = ok
            out = so.strip() if ok and so.strip() else (se.strip()[:500] if se else '(no output)')
            full += f'\n```output\n{out}\n```\n'
            
            # Continue generation
            for _ in range(TIR_MAX_RETRIES):
                try:
                    cont = vllm_chat(prompt + full, n=1, temp=0.0,
                                     max_tok=MAX_TOKENS//2, stop=['```output'])
                    chunk = cont[0] if cont else ''
                except: break
                if not chunk.strip(): break
                full += chunk
                nb = get_code_blocks(chunk)
                if nb:
                    so, se, ok = exec_code(nb[-1])
                    if ok: code_ok = True
                    out = so.strip() if ok and so.strip() else (se.strip()[:500] if se else '(err)')
                    full += f'\n```output\n{out}\n```\n'
                else: break
                if extract_boxed(full) is not None: break
        
        ans = extract_answer(full)
        w = 1.0
        if code_ok: w += 2.0
        if '\\boxed' in full: w += 0.5
        results.append((ans, w))
    
    # Weighted vote
    wt = {}
    for a, w in results:
        if a is not None: wt[a] = wt.get(a, 0.0) + w
    if not wt: return 0
    best = max(wt, key=wt.get)
    valid = [a for a, _ in results if a is not None]
    conf = Counter(valid).most_common(1)[0][1] / len(valid) if valid else 0
    print(f'  domain={domain} valid={len(valid)}/{len(results)} conf={conf:.2f} ans={best}')
    return best % ANSWER_MOD

print('Solver ready.')

In [ ]:
# Cell 7: Time manager + predict function + submit

notebook_start = time.time()
problems_remaining = 50  # approximate
solved_count = 0

def predict(id_, problem):
    global solved_count, problems_remaining
    pid = id_.item(0)
    ptxt = problem.item(0)
    
    elapsed = time.time() - notebook_start
    time_left = NOTEBOOK_LIMIT - elapsed
    reserved = max(0, problems_remaining - 1) * BASE_TIMEOUT
    budget = min(max(time_left - reserved, BASE_TIMEOUT), HIGH_TIMEOUT)
    
    print(f'\nProblem {solved_count+1} (id={pid}) budget={budget:.0f}s:')
    print(f'  {ptxt[:150]}' if len(ptxt)<=150 else f'  {ptxt[:150]}...')
    
    t0 = time.time()
    gc.disable()
    try:
        answer = solve_problem(ptxt)
    except Exception as e:
        print(f'  ERROR: {e}')
        traceback.print_exc()
        answer = 0
    gc.enable(); gc.collect()
    
    answer = int(answer) % ANSWER_MOD
    solved_count += 1
    problems_remaining = max(0, problems_remaining - 1)
    print(f'  => {answer} ({time.time()-t0:.1f}s)')
    
    return pl.DataFrame({'id': [pid], 'answer': [answer]})

# Start server
server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    server.serve()
else:
    candidates = (
        glob.glob('/kaggle/input/competitions/*/test.csv') +
        glob.glob('/kaggle/input/*/test.csv') +
        glob.glob('/kaggle/input/*/*/test.csv')
    )
    test_path = candidates[0] if candidates else '/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv'
    print(f'Test: {test_path}')
    server.run_local_gateway((test_path,))

print(f'Done! {solved_count} problems in {time.time()-notebook_start:.0f}s')